# Task 4: Model Evaluation and Deployment

This notebook covers model evaluation, TensorBoard analysis, convergence justification, and deployment readiness for both **Faster R-CNN** and **SSD**

### Model Evaluation

Run the evaluation command below to assess the trained model's performance on the test set using the TensorFlow Object Detection API. This will compute metrics like mAP and show evaluation results.

In [ ]:
# Model evaluation command
import os

BASE_PATH = r"C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1/"
OBJECT_DETECTION_PATH = fr"{BASE_PATH}/object_detection/"
MODEL_PATH = fr"{OBJECT_DETECTION_PATH}/training/TF2/training/faster_rcnn_resnet101_v1_1024x1024_coco17_tpu-8"
#MODEL_PATH = fr"{OBJECT_DETECTION_PATH}/training/TF2/training/ssd_resnet50_v1_fpn_640x640_coco17_tpu-8" # use this path for ssd model

PIPELINE_CONFIG = os.path.join(MODEL_PATH, "pipeline.config")

eval_command = (
    f'python {OBJECT_DETECTION_PATH}model_main_tf2.py '
    f'--model_dir="{MODEL_PATH}" '
    f'--pipeline_config_path="{PIPELINE_CONFIG}" '
    f'--checkpoint_dir="{MODEL_PATH}" '
    f'--alsologtostderr'
)

os.system(f'start cmd /k {eval_command}')

0

---

## Faster R-CNN Model Evaluation

The built-in COCO evaluation pipeline of the TensorFlow Object Detection API was used to do model evaluation.  
The test set that was not utilized in training (10 percent) was evaluated.  
The following indicators were calculated:

* Mean Average Precision (mAP)
* mAP at IoU = 0.50
* mAP at IoU = 0.75
* Average Recall (AR)
* RPN and Box Classifier losses

Hardware constraint meant that evaluation was done with a batch size of 1, which is customary in object detection evaluation.

### Loss Metrics

The following loss components were analysed:

* RPN Localization Loss
* RPN Objectness Loss
* Box Classifier Localization Loss
* Box Classifier Classification Loss
* Total Loss

From the TensorBoard plots:

* The loss decreases fast in the initial training. 
* After approximately 6000 steps, the loss curves become smooth and stable
* Up to 12,000 steps, only gradual improvements are observed.

This behaviour indicates stable convergence and no evidence of overfitting.

## IoU-Based Evaluation Metrics

Final evaluation results at 12,000 training steps are shown below.

### Precision (mAP)

| Metric              | Value  |
|---------------------|--------|
| mAP @ IoU 0.50:0.752| 0.752  |
| mAP @ IoU 0.500     | 0.947  |
| mAP @ IoU 0.750     | 0.897  |

Interpretation:

* The value of high mAP, at the value of the IoU = 0.50 means a high capability of object detection.
* A high score of 0.75 at IoU indicates positive and close bounding box localisation.
* The combined mAP score demonstrates overall robustness of the model.

### Recall (AR)

| Metric     | Value |
|------------|-------|
| AR @ 1     | 0.716 |
| AR @ 10    | 0.787 |
| AR @ 100   | 0.787 |
| AR @ 1000  | 0.787 |

These results show that:

* Most objects are successfully detected
* Recalling is better than the one that allows fewer detections per image.
* Missed detections are limited

## Small and Medium Object Metrics

Small and medium object categories metrics were reported to be -1.000.

Explanation:

* Only large-scale objects can be found in the dataset.
* Small or medium objects do not have ground truth annotations. 

Due to this, the COCO evaluation framework automatically does not take into consideration these categories.  
This is not an error but a behaviour that is expected.

## Training Steps (Epochs) Justification

The model was trained for 12,000 steps.

# Comparison of Evaluation Metrics: 6000 vs 12000 Steps

| Metric                                      | At 6000 Steps | At 12000 Steps | Improvement |
|---------------------------------------------|---------------|----------------|-------------|
| mAP (IoU=0.50:0.95)                          | 0.718         | **0.752**      | +0.034      |
| mAP @ IoU=0.50                              | 0.959         | **0.947**      | -0.012      |
| mAP @ IoU=0.75                              | 0.867         | **0.897**      | +0.030      |
| mAP (large objects)                         | 0.718         | **0.752**      | +0.034      |
| AR @ 1                                      | 0.692         | **0.716**      | +0.024      |
| AR @ 10                                     | 0.759         | **0.787**      | +0.028      |
| AR @ 100                                    | 0.759         | **0.787**      | +0.028      |
| AR (large objects) @ 100                    | 0.759         | **0.787**      | +0.028      |
| Total Loss                                  | 0.212         | **0.147**      | -0.065      |
| RPN Localization Loss                       | 0.091         | **0.043**      | -0.048      |
| RPN Objectness Loss                         | 0.037         | **0.030**      | -0.007      |
| Box Classifier Localization Loss            | 0.037         | **0.034**      | -0.003      |
| Box Classifier Classification Loss          | 0.047         | **0.040**      | -0.007      |

### Justification for Stopping at 12,000 Steps

Although there is a noticeable improvement in the primary COCO mAP (IoU=0.50:0.95) from 0.718 at 6000 steps to 0.752 at 12000 steps, as well as gains in mAP@0.75 and Average Recall, the overall gains are relatively low.

The total loss has decreased significantly (from 0.212 to 0.147), indicating continued learning, but the rate of improvement in key evaluation metrics has slowed considerably. Further training beyond 12,000 steps is unlikely to yield substantial additional gains, especially considering the learning rate decay in the cosine schedule, which typically leads to diminishing returns.  

Therefore, 12,000 steps were selected as the optimal stopping point to balance performance gains with training efficiency.

## Learning Rate Behaviour

A cosine decay learning rate schedule with warm-up was used.

* The warm-up phase prevents unstable gradient updates at early training stages
* Gradual decay allows fine-grained refinement of bounding boxes
* TensorBoard confirms smooth and controlled optimisation

---
---
### **SSD Model Evaluation**
---
The built-in **COCO evaluation pipeline** of the TensorFlow Object Detection API was used to evaluate the SSD model. The evaluation was performed on the **held-out test set (10%)**, which was not used during training.

The following evaluation metrics were computed:

* Mean Average Precision (mAP)
* mAP at IoU = 0.50
* mAP at IoU = 0.75
* Average Recall (AR)
* Localization, classification, and total losses

---

### **Loss Metrics**

The following loss components were analysed:

* Localization Loss
* Classification Loss
* Regularization Loss
* Total Loss

From the TensorBoard loss curves:

* Loss values decrease rapidly during the early stages of training.
* After approximately **6000 steps**, the loss curves become smooth and stable.
* From **6000 to 12000 steps**, losses continue to decrease, but only gradually.

This behaviour indicates **stable convergence**. Although training loss keeps improving, evaluation metrics show diminishing gains, suggesting the model is close to optimal learning.

---

### **IoU-Based Evaluation Metrics**

Final evaluation results were recorded at **6000 steps** and **12000 steps** to analyse performance improvement over training.

---

#### **Precision (mAP)**

##### **Results at 12,000 Steps**

| Metric              | Value |
| ------------------- | ----- |
| mAP @ IoU 0.50:0.95 | 0.744 |
| mAP @ IoU 0.50      | 0.933 |
| mAP @ IoU 0.75      | 0.869 |

**Interpretation:**

* A high **mAP@0.50** indicates strong object detection capability.
* The high **mAP@0.75** shows good bounding box localisation accuracy.
* The overall mAP confirms that the SSD model performs reliably on the dataset.

---

##### **Recall (AR)**

| Metric   | Value |
| -------- | ----- |
| AR @ 1   | 0.710 |
| AR @ 10  | 0.779 |
| AR @ 100 | 0.779 |

These results show that:

* Most objects are successfully detected.
* Recall improves when more detections per image are allowed.
* Missed detections are limited.

---

#### **Small and Medium Object Metrics**

For both evaluation checkpoints, **small and medium object metrics are reported as -1.000**.

**Explanation:**

* The dataset contains **only large objects**.
* No ground-truth annotations exist for small or medium objects.

As a result, the COCO evaluation framework excludes these categories automatically. This behaviour is **expected and not an error**.

---

### **Training Steps (Epochs) Justification**

The SSD model was trained for **12,000 steps**, with evaluations performed at **6000** and **12000 steps**.

---

### **Comparison of Evaluation Metrics: 6000 vs 12000 Steps**

| Metric                   | At 6000 Steps | At 12000 Steps | Improvement |
| ------------------------ | ------------- | -------------- | ----------- |
| mAP (IoU=0.50:0.95)      | 0.676         | **0.744**      | +0.068      |
| mAP @ IoU=0.50           | 0.893         | **0.933**      | +0.040      |
| mAP @ IoU=0.75           | 0.812         | **0.869**      | +0.057      |
| mAP (large objects)      | 0.676         | **0.744**      | +0.068      |
| AR @ 1                   | 0.663         | **0.710**      | +0.047      |
| AR @ 10                  | 0.710         | **0.779**      | +0.069      |
| AR @ 100                 | 0.710         | **0.779**      | +0.069      |
| AR (large objects) @ 100 | 0.710         | **0.779**      | +0.069      |
| Total Loss               | 0.457         | **0.401**      | -0.056      |
| Localization Loss        | 0.075         | **0.057**      | -0.018      |
| Classification Loss      | 0.154         | **0.146**      | -0.008      |
| Regularization Loss      | 0.228         | **0.199**      | -0.029      |

---

### **Justification for Stopping at 12,000 Steps**

The SSD model shows clear improvement from **6000 to 12000 steps**, particularly in overall mAP and Average Recall.
However, although training loss continues to decrease, the **rate of improvement in evaluation metrics slows down significantly** after 6000 steps.

This indicates diminishing returns from further training. Continuing training beyond 12000 steps is unlikely to result in substantial performance gains and may increase the risk of overfitting, especially given the cosine learning rate decay schedule.

Therefore, **12,000 training steps** were selected as the optimal stopping point, balancing performance improvement with training efficiency.


---

## **Faster R-CNN vs SSD Evaluation Comparison (12,000 Steps)**

| Metric                | Faster R-CNN  | SSD        |
| --------------------- | ------------- | ---------- |
| mAP (IoU = 0.50:0.95) | **0.752**     | 0.744      |
| mAP @ IoU = 0.50      | **0.947**     | 0.933      |
| mAP @ IoU = 0.75      | **0.897**     | 0.869      |
| AR @ 1                | **0.716**     | 0.710      |
| AR @ 10               | **0.787**     | 0.779      |
| AR @ 100              | **0.787**     | 0.779      |
| Total Loss            | **0.147**     | 0.401      |
| Best at Localization  | **Yes**       | Good       |
| Training Stability    | **Very High** | High       |
| Inference Speed       | Slower        | **Faster** |

## **Interpretation**

* **Faster R-CNN** achieves **higher accuracy and better localisation**, especially at stricter IoU (0.75).
* **SSD** is slightly less accurate but offers **faster inference**, making it suitable for real-time or resource-limited scenarios.
* Since the dataset contains **only large objects**, both models perform reliably, but **Faster R-CNN is overall stronger** in precision-focused tasks.


---

# Model Deployment 

In [ ]:
OUTPUT_DIR = os.path.join(MODEL_PATH, "saved_model") # Directory to save the exported model
# Export the trained model using the exporter script
!python {OBJECT_DETECTION_PATH}exporter_main_v2.py --input_type image_tensor --pipeline_config_path "{PIPELINE_CONFIG}" --trained_checkpoint_dir "{MODEL_PATH}" --output_directory "{OUTPUT_DIR}"

## Command Purpose

This command exports a **trained TensorFlow Object Detection model** from training checkpoints into a **SavedModel** format that can be used for inference and deployment.

## Parameter Explanations

### 1. `exporter_main_v2.py`

This is the official TensorFlow Object Detection API script used to export a trained model. It converts the model from training checkpoints into a TensorFlow **SavedModel** format.

---

### 2. `--input_type image_tensor`

Specifies the input format that the exported model expects during inference.

* `image_tensor` means the model will accept a TensorFlow image tensor as input.
* The input shape is typically:

  ```
  [1, height, width, 3]
  ```
* This format is commonly used for real-time inference and Python-based prediction pipelines.

---

### 3. `--pipeline_config_path`

Provides the path to the `pipeline.config` file. During export, this configuration is used to rebuild the model structure before loading the trained weights.

---

### 4. `--trained_checkpoint_dir`

Specifies the directory containing the trained model checkpoints.

This directory typically includes files such as:

```
ckpt-XXXX.index
ckpt-XXXX.data-00000-of-00001
checkpoint
```

The exporter automatically selects the **latest checkpoint** and loads its learned weights into the model.

---

### 5. `--output_directory`

Defines the directory where the exported model will be saved.

The output is stored in **TensorFlow SavedModel format**, usually with the following structure:

```
saved_model/
     saved_model.pb
```

This exported model can be directly loaded for inference using TensorFlow.

---
